### How many taxi trips will occur in each zone, each hour?


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import logging, os, warnings
warnings.filterwarnings("ignore")

In [2]:
builder = (
    SparkSession.builder
        .appName("ForecastDemand")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [3]:
log_dir = "../../logs"
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(log_dir, "forecast_demand_model_training.log"),
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
df = (
    spark.read.format("delta")
    .load(r"..\..\04_storage_platinum\demand_series")
    .toPandas()
)

In [5]:
# Build proper datetime column for sorting
df["ds"] = (
    pd.to_datetime(df["trip_date"]) +
    pd.to_timedelta(df["pickup_hour"], unit="h")
)
df = df.sort_values(["pu_location_id", "ds"]).reset_index(drop=True)

logging.info(f"Loaded demand_series: {len(df)} rows, {df['pu_location_id'].nunique()} zones")


In [6]:
# --------------------------------------------------
# TRAIN / TEST SPLIT
# Last 30 days = test (holdout); everything else = train
# This simulates a real deployment: model trained on
# historical data, evaluated on unseen future dates.
# --------------------------------------------------

CUTOFF     = df["ds"].max() - pd.Timedelta(days=30)
train_df   = df[df["ds"] <= CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])
test_df    = df[df["ds"] >  CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])

logging.info(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")

In [7]:
# --------------------------------------------------
# FEATURES
# zone_id is passed as categorical — LightGBM learns
# per-zone demand level without one-hot encoding,
# which would explode dimensionality at 265 zones.
# --------------------------------------------------

FEATURE_COLS = [
    "pu_location_id",      # categorical — zone identity
    "pickup_hour",
    "day_of_week",
    "month",
    "trip_year",
    "is_weekend",
    "is_rush_hour",
    "is_night",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "avg_fare",
    "avg_distance",
    "avg_passengers",
]
TARGET = "trip_count"

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

### LIGHTGBM — GLOBAL MODEL
##### One model, all zones. pu_location_id is categorical.

In [8]:
lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=["pu_location_id"],
    free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_test, label=y_test,
    categorical_feature=["pu_location_id"],
    reference=lgb_train,
    free_raw_data=False
)

In [9]:
params = {
    "objective":        "regression_l1",   # MAE loss — robust to demand spikes
    "metric":           ["mae", "rmse"],
    "learning_rate":    0.05,
    "num_leaves":       127,               # 2^7 — balanced depth for tabular data
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "verbose":          -1,
    "n_jobs":           -1,
    "seed":             42,
}

In [10]:
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100)
]

In [11]:
logging.info("Training global LightGBM model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=callbacks
)
logging.info(f"Best iteration: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds
[100]	train's l1: 7.77088	train's rmse: 16.7383	val's l1: 7.99373	val's rmse: 17.8174
[200]	train's l1: 7.21012	train's rmse: 15.0753	val's l1: 7.50811	val's rmse: 16.158
[300]	train's l1: 6.97595	train's rmse: 14.4787	val's l1: 7.30912	val's rmse: 15.6164
[400]	train's l1: 6.83035	train's rmse: 14.1149	val's l1: 7.19049	val's rmse: 15.32
[500]	train's l1: 6.73208	train's rmse: 13.9195	val's l1: 7.12115	val's rmse: 15.1857
[600]	train's l1: 6.64616	train's rmse: 13.7247	val's l1: 7.0607	val's rmse: 15.0471
[700]	train's l1: 6.58028	train's rmse: 13.5857	val's l1: 7.01547	val's rmse: 14.9502
[800]	train's l1: 6.5295	train's rmse: 13.4812	val's l1: 6.98237	val's rmse: 14.8864
[900]	train's l1: 6.48116	train's rmse: 13.3763	val's l1: 6.95102	val's rmse: 14.8164
[1000]	train's l1: 6.43671	train's rmse: 13.2905	val's l1: 6.92469	val's rmse: 14.7655
Did not meet early stopping. Best iteration is:
[999]	train's l1: 6.43693	train's r

### Evaluation

Global

In [12]:
preds = np.clip(model.predict(X_test), 0, None)

mae  = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)
r2   = r2_score(y_test, preds)
mape = np.mean(
    np.abs((y_test.values - preds) /
           np.where(y_test.values == 0, 1, y_test.values))
) * 100

print("\n" + "="*45)
print("GLOBAL MODEL — TEST SET PERFORMANCE")
print("="*45)
print(f"  MAE  : {mae:.2f} trips/hour")
print(f"  RMSE : {rmse:.2f} trips/hour")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
logging.info(f"Global — MAE={mae:.2f} RMSE={rmse:.2f} R²={r2:.4f} MAPE={mape:.2f}%")


GLOBAL MODEL — TEST SET PERFORMANCE
  MAE  : 6.92 trips/hour
  RMSE : 14.77 trips/hour
  R²   : 0.9650
  MAPE : 33.87%


Demand based MAPE

In [17]:
# Split MAPE by demand level — shows model is accurate where it matters
test_df["demand_bucket"] = pd.cut(
    test_df["trip_count"],
    bins=[0, 10, 50, 200, float("inf")],
    labels=["very low (<10)", "low (10–50)", "medium (50–200)", "high (200+)"]
)

mape_by_bucket = (
    test_df.groupby("demand_bucket")
    .apply(lambda g: np.mean(
        np.abs((g["trip_count"] - g["prediction"]) /
               np.where(g["trip_count"] == 0, 1, g["trip_count"])) * 100
    ))
    .reset_index(name="mape")
)
print(mape_by_bucket)

     demand_bucket       mape
0   very low (<10)  49.773174
1      low (10–50)  24.112540
2  medium (50–200)  13.808080
3      high (200+)  11.044531


Per - Zone

In [13]:
test_df = test_df.copy()
test_df["prediction"] = preds

per_zone = []
for zone_id, grp in test_df.groupby("pu_location_id"):
    y_z  = grp[TARGET].values
    p_z  = grp["prediction"].values
    per_zone.append({
        "zone_id":  zone_id,
        "mae":      round(mean_absolute_error(y_z, p_z), 2),
        "rmse":     round(root_mean_squared_error(y_z, p_z), 2),
        "r2":       round(r2_score(y_z, p_z), 4),
        "mape":     round(np.mean(np.abs((y_z - p_z) /
                         np.where(y_z == 0, 1, y_z))) * 100, 2)
    })

per_zone_df = pd.DataFrame(per_zone).sort_values("mae")
print("\nPer-zone results (sorted by MAE):")
print(per_zone_df.to_string(index=False))


Per-zone results (sorted by MAE):
 zone_id   mae  rmse     r2  mape
      93  0.35  0.64 0.2297 23.83
      10  0.82  1.28 0.4997 33.70
      12  0.87  1.36 0.6383 37.84
     260  0.93  1.31 0.4048 45.98
     216  0.96  1.46 0.3499 42.57
      82  0.99  1.48 0.3520 46.87
     223  1.06  1.75 0.3915 44.88
     197  1.07  1.75 0.3405 48.35
      95  1.10  1.61 0.3477 46.02
      71  1.15  1.66 0.5280 43.86
     179  1.16  1.68 0.3750 49.70
      72  1.21  1.67 0.5529 46.51
      91  1.26  1.86 0.4858 43.75
     168  1.27  1.83 0.3533 50.39
     177  1.29  1.84 0.4961 47.90
     146  1.31  1.99 0.4520 48.66
     129  1.36  2.00 0.4793 46.93
     130  1.36  1.94 0.3932 50.85
      35  1.47  2.11 0.4664 49.32
      49  1.51  2.21 0.4388 47.82
      39  1.62  2.36 0.5602 51.69
      25  1.62  2.27 0.5640 49.49
     226  1.62  2.27 0.6544 42.43
      89  1.63  2.32 0.5561 44.88
      17  1.66  2.33 0.4939 53.21
      65  1.67  2.43 0.5882 48.78
      36  1.71  4.13 0.7356 49.35
     152  1.7

Feature Importance

In [14]:
importance_df = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False)

print("\nFeature importances (gain):")
print(importance_df.to_string(index=False))
logging.info("Feature importances:\n" + importance_df.to_string())


Feature importances (gain):
       feature   importance
        lag_1h 2.762657e+07
       lag_24h 7.759318e+06
pu_location_id 3.584730e+06
avg_passengers 3.075165e+06
   pickup_hour 2.038865e+06
      lag_168h 1.313950e+06
  avg_distance 1.244723e+06
      avg_fare 1.085150e+06
   day_of_week 5.333718e+05
      is_night 4.809610e+05
     trip_year 3.694080e+05
    is_weekend 2.826822e+05
         month 1.475256e+05
  is_rush_hour 5.636901e+04


### Save Outputs to Platinum (For Forecasting Dashboard)

In [15]:
# Scored predictions
(
    spark.createDataFrame(test_df[
        ["pu_location_id", "ds", "trip_count", "prediction",
         "is_rush_hour", "is_weekend", "pickup_hour"]
    ].rename(columns={"ds": "datetime"}))
    .write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(r"..\forecasting_demand\dashboard_data\demand_predictions")
)

# Per-zone metrics
(
    spark.createDataFrame(per_zone_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\dashboard_data\forecast_metrics")
)

# Feature importances
(
    spark.createDataFrame(importance_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\dashboard_data\forecast_feature_importance")
)

logging.info("All platinum outputs written. Forecast complete.")

### Save model for future inference (e.g. batch scoring, real-time API)

In [16]:
import joblib

model_package = {
    "model": model,
    "features": FEATURE_COLS
}

model_dir = r"..\forecasting_demand\model"

model_path = os.path.join(model_dir, "demand_forecast_lgbm.pkl")

joblib.dump(model_package, model_path)

logging.info(f"Model saved to {model_path}")
print(f"\nModel saved to {model_path}")


Model saved to ..\forecasting_demand\model\demand_forecast_lgbm.pkl
